<a href="https://colab.research.google.com/github/oerv13-gh/Ibero/blob/main/SMA_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# === PAYMENT GATEWAY MULTI-AGENT SYSTEM ===

!pip install pyyaml -q

import asyncio
import uuid
from enum import Enum
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Callable
from datetime import datetime

# ============= PROTOCOLO FIPA-ACL =============
class ACLPerformative(Enum):
    REQUEST = "request"; INFORM = "inform"; CONFIRM = "confirm"

class MessageType(Enum):
    TRANSACTION_REQUEST = "transaction-request"
    BIOMETRIC_DATA = "biometric-data"
    RISK_ASSESSMENT = "risk-assessment"
    INFRASTRUCTURE_STATUS = "infrastructure-status"
    TRANSACTION_APPROVAL = "transaction-approval"
    TRANSACTION_REJECTION = "transaction-rejection"

class AgentRole(Enum):
    USER_AGENT = "ua"; RISK_AGENT = "ra"; INFRASTRUCTURE_AGENT = "ia"; COORDINATOR_AGENT = "ca"

@dataclass
class Message:
    performative: ACLPerformative; sender: str; receivers: List[str]
    content: Dict[str, Any]; message_type: MessageType
    conversation_id: str = field(default_factory=lambda: str(uuid.uuid4()))

class MessageHandler:
    def __init__(self):
        self._queues: Dict[str, asyncio.Queue] = {}
        self._handlers: Dict[str, List[Callable]] = {}
    def register_agent(self, aid):
        self._queues[aid] = asyncio.Queue()
        print(f"[MH] Registered agent: {aid}") # ADDED DEBUG PRINT
    async def send(self, msg):
        print(f"[MH] Attempting to send {msg.message_type.value} from {msg.sender} to {msg.receivers}") # ADDED DEBUG PRINT
        for r in msg.receivers:
            if r in self._queues:
                await self._queues[r].put(msg)
                print(f"[MH] Successfully put {msg.message_type.value} for {r}") # ADDED DEBUG PRINT
            else:
                print(f"[MH] ERROR: Receiver {r} not registered in _queues") # ADDED DEBUG PRINT
            if r in self._handlers:
                for h in self._handlers[r]: await h(msg)
    async def receive(self, aid, timeout=5):
        if aid not in self._queues: return None
        try: return await asyncio.wait_for(self._queues[aid].get(), timeout=timeout)
        except: return None

# ============= BASE AGENT =============
class BaseAgent:
    def __init__(self, aid, role, mh):
        self.aid, self.role, self.mh = aid, role, mh
        self._running = False
    @property
    def name(self): return f"{self.role.value}_{self.aid}"
    async def setup(self):
        self.mh.register_agent(self.name)
        print(f"[{self.name}] ✓ Inicializado")
    async def start(self):
        self._running = True
        await self.setup()
        asyncio.create_task(self._loop())
        print(f"[{self.name}] ✓ Activo")
    async def stop(self):
        self._running = False
        # Optional: Add a small delay to allow pending tasks to complete if necessary
        # await asyncio.sleep(0.1)
    async def _loop(self):
        print(f"[{self.name}] _loop started.") # ADDED DEBUG PRINT
        while self._running:
            # print(f"[{self.name}] _loop is running, waiting for message...") # Too noisy
            msg = await self.mh.receive(self.name, 1)
            if msg:
                print(f"[{self.name}] _loop received message: {msg.message_type.value}") # ADDED DEBUG PRINT
                await self.process(msg)
            else:
                pass # print(f"[{self.name}] _loop received None (idle/timeout)") # Too noisy
    async def send(self, to, perf, mtype, content):
        msg = Message(performative=perf, sender=self.name, receivers=to, content=content, message_type=mtype)
        await self.mh.send(msg)
        print(f"  [{self.name}] → {to}: {mtype.value}")

# ============= USER AGENT =============
class UserAgent(BaseAgent):
    def __init__(self, aid, mh):
        super().__init__(aid, AgentRole.USER_AGENT, mh)
    async def process(self, msg):
        # User Agent now only processes final decisions from the Coordinator Agent
        if msg.message_type == MessageType.TRANSACTION_APPROVAL:
            print(f"[{self.name}] ✓ Transacción {msg.content.get('transaction_id')} {msg.content.get('decision')}")
        elif msg.message_type == MessageType.TRANSACTION_REJECTION:
            print(f"[{self.name}] ✗ Transacción {msg.content.get('transaction_id')} {msg.content.get('decision')}")

# ============= RISK AGENT =============
class RiskAgent(BaseAgent):
    def __init__(self, aid, mh):
        super().__init__(aid, AgentRole.RISK_AGENT, mh)
    async def process(self, msg):
        if msg.message_type == MessageType.BIOMETRIC_DATA:
            tx_id = msg.content.get("transaction_id")
            amount = msg.content.get("amount", 0)
            risk = 0.1 # Base risk
            if amount > 10000:
                risk += 0.5 # High risk for very large transactions
            elif amount > 5000:
                risk += 0.3 # Medium risk for large transactions
            else:
                risk += 0.1 # Low risk for small transactions
            risk = min(1.0, risk) # Ensure risk doesn't exceed 1.0
            print(f"[{self.name}] Evaluando riesgo: {tx_id} → Score: {risk:.2f}")
            await self.send(["ca"], ACLPerformative.INFORM, MessageType.RISK_ASSESSMENT, {
                "transaction_id": tx_id, "risk_score": risk, "recommendation": "approve" if risk < 0.5 else "review"
            })

# ============= INFRASTRUCTURE AGENT =============
class InfrastructureAgent(BaseAgent):
    def __init__(self, aid, mh):
        super().__init__(aid, AgentRole.INFRASTRUCTURE_AGENT, mh)
    async def process(self, msg):
        if msg.message_type == MessageType.TRANSACTION_REQUEST:
            tx_id = msg.content.get("transaction_id")
            health = 0.95
            print(f"[{self.name}] Verificando infraestructura: {tx_id} → Health: {health:.2f}")
            await self.send(["ca"], ACLPerformative.INFORM, MessageType.INFRASTRUCTURE_STATUS, {
                "transaction_id": tx_id, "network_health": health
            })

# ============= COORDINATOR AGENT =============
class CoordinatorAgent(BaseAgent):
    def __init__(self, aid, mh): super().__init__(aid, AgentRole.COORDINATOR_AGENT, mh); self.tx = {}
    async def process(self, msg):
        tx_id = msg.content.get("transaction_id")
        if msg.message_type == MessageType.TRANSACTION_REQUEST:
            self.tx[tx_id] = {"risk": None, "infra": None}
            print(f"[{self.name}] Transacción recibida: {tx_id}")
        elif msg.message_type == MessageType.RISK_ASSESSMENT:
            self.tx[tx_id]["risk"] = msg.content.get("risk_score")
            await self.decide(tx_id)
        elif msg.message_type == MessageType.INFRASTRUCTURE_STATUS:
            self.tx[tx_id]["infra"] = msg.content.get("network_health")
            await self.decide(tx_id)
    async def decide(self, tx_id):
        t = self.tx.get(tx_id)
        if not t or t["risk"] is None or t["infra"] is None: return
        if t["risk"] < 0.3 and t["infra"] > 0.7: decision = "APPROVED"
        elif t["risk"] < 0.5: decision = "APPROVED_WITH_MONITORING"
        else: decision = "UNDER_REVIEW"
        print(f"[{self.name}] ★ DECISIÓN: {decision} (risk={t['risk']:.2f}, infra={t['infra']:.2f})")
        await self.send(["ua"], ACLPerformative.INFORM, MessageType.TRANSACTION_APPROVAL if "APPROVED" in decision else MessageType.TRANSACTION_REJECTION, {"transaction_id": tx_id, "decision": decision})

# ============= DEMO =============
async def run_demo():
    print("="*60)
    print("PAYMENT GATEWAY MULTI-AGENT SYSTEM")
    print("="*60)

    mh = MessageHandler()
    ca = CoordinatorAgent("CA", mh)
    ia = InfrastructureAgent("IA", mh)
    ra = RiskAgent("RA", mh)
    ua = UserAgent("UA", mh)

    for a in [ca, ia, ra, ua]: await a.start()
    await asyncio.sleep(0.5)

    print("\n>>> EJECUTANDO TRANSACCIONES DE PRUEBA")
    print("-"*60)

    # Test cases: (amount, customer_id, merchant_id)
    # 1. APPROVED (risk < 0.3): amount <= 5000 --> risk = 0.2
    # 2. APPROVED_WITH_MONITORING (risk < 0.5): 5000 < amount <= 10000 --> risk = 0.4
    # 3. UNDER_REVIEW (risk >= 0.5): amount > 10000 --> risk = 0.6
    tests = [
        (150.00, "CUST001", "STORE001"), # Should be APPROVED (risk=0.2)
        (7500.00, "CUST002", "LUXURY"),   # Should be APPROVED_WITH_MONITORING (risk=0.4)
        (12000.00, "CUST003", "BIGSHOP")   # Should be UNDER_REVIEW (risk=0.6)
    ]
    for amt, cust, merch in tests:
        tx_id = f"TX-{uuid.uuid4().hex[:6].upper()}"
        print(f"\n>>> Nueva: ${amt:.2f} | {cust} → {merch} (ID: {tx_id})")

        # User Agent (as initiator) sends request components to relevant agents
        # 1. Biometric data to Risk Agent
        await ua.send([ra.name], ACLPerformative.INFORM, MessageType.BIOMETRIC_DATA, {
            "transaction_id": tx_id,
            "amount": amt,
            "biometric_score": 0.92
        })
        # 2. Transaction details to Infrastructure Agent
        await ua.send([ia.name], ACLPerformative.REQUEST, MessageType.TRANSACTION_REQUEST, {
            "transaction_id": tx_id,
            "amount": amt,
            "customer_id": cust,
            "merchant_id": merch
        })
        # 3. Transaction details to Coordinator Agent
        await ua.send([ca.name], ACLPerformative.REQUEST, MessageType.TRANSACTION_REQUEST, {
            "transaction_id": tx_id,
            "amount": amt,
            "customer_id": cust,
            "merchant_id": merch
        })

        await asyncio.sleep(15) # Increased sleep time to 15 seconds

    print("\n" + "="*60)
    print("SISTEMA DETENIDO")
    print("="*60)
    for a in [ua, ra, ia, ca]: await a.stop()

# EJECUTAR
await run_demo()


PAYMENT GATEWAY MULTI-AGENT SYSTEM
[MH] Registered agent: ca_CA
[ca_CA] ✓ Inicializado
[ca_CA] ✓ Activo
[MH] Registered agent: ia_IA
[ia_IA] ✓ Inicializado
[ia_IA] ✓ Activo
[MH] Registered agent: ra_RA
[ra_RA] ✓ Inicializado
[ra_RA] ✓ Activo
[MH] Registered agent: ua_UA
[ua_UA] ✓ Inicializado
[ua_UA] ✓ Activo
[ca_CA] _loop started.
[ia_IA] _loop started.
[ra_RA] _loop started.
[ua_UA] _loop started.

>>> EJECUTANDO TRANSACCIONES DE PRUEBA
------------------------------------------------------------

>>> Nueva: $150.00 | CUST001 → STORE001 (ID: TX-BBA263)
[MH] Attempting to send biometric-data from ua_UA to ['ra_RA']
[MH] Successfully put biometric-data for ra_RA
  [ua_UA] → ['ra_RA']: biometric-data
[MH] Attempting to send transaction-request from ua_UA to ['ia_IA']
[MH] Successfully put transaction-request for ia_IA
  [ua_UA] → ['ia_IA']: transaction-request
[MH] Attempting to send transaction-request from ua_UA to ['ca_CA']
[MH] Successfully put transaction-request for ca_CA
  [ua_UA